# 4.1 从基础算子走向模型模块

学完基础算子后，下一步不是记住更多 API，而是学会把已有能力组合起来。真实模型里的一个模块，往往同时包含逐元素计算、归约、矩阵乘、shape 变换、动态维度处理和多 kernel 调用。

这一节先把学习方法和运行环境准备好。后面会从激活函数、Softmax、LayerNorm、RMSNorm、FFN 一直走到 Attention、Transformer block、Cost Model、ACLGraph 和章节综合实践。每一节都会围绕同一个闭环展开：先明确要计算什么，再拆 shape 和公式，然后写 PyPTO kernel，最后用 PyTorch reference 对照验证。

## 1. 从这里开始会遇到什么变化

先看后面几节会做什么，心里有一张地图，读代码时就不容易被中间 Tensor 和 shape 变换绕住：

| 章节 | 主要内容 | 你会重点练到什么 |
| --- | --- | --- |
| 4.2 | 自定义激活函数与 Softmax | SiLU、GELU、SwiGLU、GeGLU、稳定 Softmax，以及逐元素计算和归约计算的组合方式。 |
| 4.3 | LayerNorm、RMSNorm 与 FFN | 均值/方差归一化、RMS 归一化、前馈网络里的 matmul + activation + matmul 数据流。 |
| 4.4 | 动态 Shape 与控制流 | `pypto.DYNAMIC`、`view`、`valid_shape`、`assemble`、`pypto.loop` 和条件分支。 |
| 4.5 | Attention 与 Transformer 组合 | `Q @ K^T -> softmax -> @ V`、Q/K/V 投影、多头拆分/合并、残差连接和多 kernel 组合。 |
| 4.6 | 系统分析与加速 | Cost Model 输出分析、ACLGraph 图捕获与 replay，理解算子如何进入系统优化链路。 |
| 4.7 | 章节实践 | 综合使用激活函数、归一化、动态 shape、loop、view 和 assemble，完成可验证的融合算子。 |

进入中高级实践后，代码变长的原因通常不是某个 API 变复杂，而是一个模块里同时出现了多种基础能力。阅读时可以先抓住这些复用关系：

- Softmax 既可以单独实现，也会成为 Attention 的核心步骤。
- dynamic shape 既能单独练习，也会被动态 Attention 和动态 FFN 使用。
- LayerNorm、GELU、Residual Connection 单独看是小模块，组合起来就是 Transformer block。
- Cost Model 和 ACLGraph 不改变数学结果，但会影响分析、集成和执行效率。

也就是说，后面的内容不是互不相关的新知识点，而是在不断复用和放大前面学过的模式：先组合算子，再组合模块，最后理解系统分析和图捕获，并通过章节实践检查自己能否把多类能力组合到同一个 kernel 中。

## 2. 先确认运行环境

先运行下面这个单元。它不会实现具体算子，只负责把 Notebook 调到一个稳定状态：导入依赖、清理 PyPTO 记录状态、选择 CPU / NPU 设备，并设置 `RUN_MODE`。

In [ ]:
import os
os.environ['TILE_FWK_DEVICE_ID'] = '0'
import torch
import pypto
import torch_npu


def get_device():
    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    return f"npu:{device_id}"


device = get_device()
RUN_MODE = pypto.RunMode.NPU
print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<not set>"))
print("device:", device)
print("run_mode:", RUN_MODE)
print("pypto:", pypto.__file__)


### 2.1 这段环境代码在做什么

运行环境单元时，重点看清每一层的职责：

| 代码片段 | 所在层次 | 作用 |
| --- | --- | --- |
| `import torch` | Host 侧 | 创建真实输入、输出和 PyTorch reference |
| `import pypto` | PyPTO 编译入口 | 定义 JIT kernel、Tensor 描述和算子表达式 |
| `import torch_npu` | NPU 运行环境 | 当前环境支持 NPU 时启用真实设备执行 |
| `get_device()` | 设备选择 | 有 NPU 时返回 `npu:设备号`，否则返回 `cpu` |
| `RUN_MODE` | JIT 配置 | 有 NPU 时用 `pypto.RunMode.NPU`，否则用 `pypto.RunMode.SIM` |

其中最容易混淆的是 `torch.Tensor` 和 `pypto.Tensor`：

- `torch.Tensor` 是真实数据，存在于 CPU 或 NPU 上。
- `pypto.Tensor(...)` 是 kernel 参数描述，告诉 PyPTO 输入输出长什么样。

后面写任何 kernel 时，都可以沿用这个分工：Host 侧准备真实数据，kernel 侧描述计算图。

## 3. 拿到一段中高级代码时先看什么

中高级代码通常包含更多中间 Tensor 和 shape 变换。不要急着逐行抠细节，先按下面六步建立整体轮廓：

1. 看数学目标：这一段最终要算什么。
2. 看输入输出 shape：输入是什么，输出是什么，中间 shape 如何变化。
3. 看核心 PyPTO 表达式：哪些是 `matmul`，哪些是 `sum / amax`，哪些是 `reshape / transpose`。
4. 看 tile 配置：用 `set_vec_tile_shapes` 还是 `set_cube_tile_shapes`。
5. 看数据组织：是否使用 `view`、`assemble`、`loop`、`valid_shape`。
6. 看验证闭环：PyTorch reference 是怎么写的，误差阈值怎么判断。

这样读，Attention、FFN、动态 shape 和 ACLGraph 都不会显得杂乱。它们只是把前面学过的基础部件放进了更大的结构里。

## 4. 先用一个小 kernel 热身

后面每个模块都会比这个例子更长，但基本执行闭环是一致的：

```text
Host 侧准备输入
  -> JIT kernel 描述计算
  -> 写回输出 Tensor
  -> PyTorch reference 验证
```

下面先用一个很小的二元组合算子热身。它只做 `add + relu`，目的是把 Host 输入、JIT kernel、输出写回和 reference 验证这四步连起来。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def fused_add_relu_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    bias: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(8, 8)
    y = x + bias
    out.move(pypto.maximum(y, 0.0))


def main_fused_add_relu():
    x = torch.randn((8, 8), dtype=torch.float32, device=device)
    bias = torch.randn((8, 8), dtype=torch.float32, device=device)
    out = torch.empty(x.shape, dtype=x.dtype, device=device)

    fused_add_relu_kernel(x, bias, out)
    ref = torch.maximum(x + bias, torch.zeros_like(x))

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("fused_add_relu_kernel 验证通过")
    print("输入 shape:", tuple(x.shape), "输出 shape:", tuple(out.shape))
    print("最大误差:", max_diff)


main_fused_add_relu()


### 4.1 最小示例逐行理解

这个小 kernel 可以翻译成自然语言：

```text
接收 x 和 bias。
先逐元素相加得到 y。
取 y 和标量 0 的逐元素较大值，实现 ReLU。
把结果写入 out。
```

逐段看代码：

| 代码 | 含义 |
| --- | --- |
| `@pypto.frontend.jit(...)` | 标记这个函数进入 PyPTO 编译流程 |
| `pypto.Tensor([], pypto.DT_FP32)` | 参数是 FP32 Tensor，实际 shape 运行时由传入的 `torch.Tensor` 决定 |
| `pypto.set_vec_tile_shapes(8, 8)` | 这是向量类计算，使用 vec tile 组织执行 |
| `y = x + bias` | 生成逐元素加法 Operation |
| `pypto.maximum(y, 0.0)` | 实现 `max(y, 0)`，这里直接使用标量 0，避免为最小示例额外创建全 0 Tensor |
| `out.move(...)` | 把计算结果写回 Host 侧传入的输出 Tensor |

后面看到更大的模块时，也可以从这个小例子的角度类比：激活函数主要是 elementwise 组合，LayerNorm 会多出 reduction，FFN 会多出 matmul，Attention 会多出 transpose / reshape / softmax。

## 5. 学完后你应该能够

完成 4.2 到 4.7 后，可以回到这里检查自己是否已经做到：

1. 说清楚激活函数、Softmax、归一化、FFN、Attention 分别解决什么问题。
2. 看懂 `pypto.DYNAMIC`、`pypto.loop`、`valid_shape` 和条件分支在 kernel 里的作用。
3. 理解 `view`、`assemble`、`transpose`、`reshape` 如何组织数据流。
4. 把多个基础算子组合成可验证的中级和高级网络片段。
5. 能够用 PyTorch reference 对照验证 PyPTO 实现。
6. 理解 Cost Model 和 ACLGraph 这类系统能力与数学算子的区别。
7. 能把激活、归一化、动态 shape 和分块写回组合成一个章节综合实践算子。

## 6. 建议学习顺序

建议仍然按 4.2 到 4.7 顺序阅读：先会写组合算子，再会写归一化和 FFN 这类模块，然后处理动态 shape 和控制流，接着进入 Attention / Transformer 组合，最后理解系统分析与图捕获，并通过章节实践把这些能力串成一个完整任务。这个顺序的好处是每一节都会复用上一节的能力，复杂度是逐步叠上去的。


## 7. 每一步为什么接在下一步前面

| 学习阶段 | 解决的问题 | 后续承接 |
| --- | --- | --- |
| 激活函数与 Softmax | 如何把逐元素计算、指数、求和和归一化组合成稳定公式 | 为归一化、FFN 和 Attention 准备基础算子组合能力 |
| LayerNorm、RMSNorm、FFN | 如何把 reduction、matmul 和激活函数组织成神经网络子模块 | 为 Transformer block 的前馈分支做准备 |
| dynamic shape、loop、condition | 如何在输入维度变化时稳定地切块、计算和写回 | 为动态 batch、动态 attention 和系统集成做准备 |
| Attention 与 Transformer 组合 | 如何组织 Q/K/V、多头拆分、残差连接和多 kernel 调用 | 把前面的小模块组合成完整模型片段 |
| Cost Model 与 ACLGraph | 如何观察执行成本，并捕获可重放的执行图 | 从算子正确性走向性能分析和工程集成 |
| 章节实践 | 如何把动态 shape、归一化、激活和输出写回组合成完整算子 | 检查第四章知识是否能独立迁移到综合任务 |

## 8. 先认几个高频术语

| 术语 | 直观理解 | 后续出现位置 |
| --- | --- | --- |
| Operator Composition | 把多个基础算子拼成新算子 | 4.2、4.3 |
| Reduction | 沿某个维度把一组数归并成较少的数 | 4.2、4.3 |
| Dynamic Shape | 某些维度运行时才确定 | 4.4、4.5、4.6 |
| View / Assemble | 从大 Tensor 取局部块，再写回大 Tensor | 4.4、4.5 |
| Multi-head | 把 hidden 拆成多个 head 并行计算 | 4.5 |
| Residual Connection | 把输入直接加回输出，形成残差路径 | 4.5 |
| Cost Model | 模拟和分析执行成本 | 4.6 |
| ACLGraph | 捕获执行图并重放，减少重复调度开销 | 4.6 |
| Chapter Practice | 用一个综合任务检查本章能力 | 4.7 |

## 9. 课后练习

本节练习用于检查中高级实践的整体阅读方法。请结合本节的最小 `add + relu` kernel 和后续章节路线完成以下题目。

1. （选择题）为什么学习时要先理解能力之间的复用关系？  
   A. 因为中高级能力会在 Softmax、dynamic shape、Attention 等场景中反复组合出现  
   B. 因为每个主题都完全独立  
   C. 因为可以不看 shape 直接写代码  
   D. 因为 Cost Model 会改变数学公式
2. （填空题）`torch.Tensor` 处在________侧，`pypto.Tensor` 是 kernel 的________。
3. （选择题）阅读一个中高级 kernel 时，为什么要先看 shape？  
   A. shape 决定 matmul、transpose、reshape、reduction 和 broadcast 是否能对齐  
   B. shape 可以替代所有验证逻辑  
   C. shape 会自动修复错误 dtype  
   D. shape 与中高级 kernel 无关
4. （填空题）`view -> compute -> assemble` 这个模式通常用于解决________问题。
5. （选择题）Cost Model 和 ACLGraph 会改变 Softmax 的数学含义吗？  
   A. 会  
   B. 不会

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/04.01_answer.txt


## 10. 小结

这一节先完成三件事：确认运行环境，建立阅读中高级 kernel 的顺序，并用一个最小 `add + relu` kernel 回顾完整验证闭环。

接下来进入具体代码：先学习如何把基础算子组合成 SiLU、GELU、SwiGLU、GeGLU 和稳定 Softmax，最后在 4.7 中完成章节综合实践。